In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import datetime

# Cấu hình giao diện đồ thị
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
path = r"E:\Air Quality\air_quality.csv"
df = pd.read_csv(path)

#Extract date and hour
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['date'] = df['timestamp'].dt.date

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
unique_stations = df['station_name'].unique()

print(unique_stations)

In [ ]:
hanoi_stations = [
    # 'ĐH Bách Khoa - cổng Parabol đường Giải Phóng',
    # 'Hà Nội: Công viên hồ điều hòa Nhân Chính  Khuất Duy Tiến (KK)',
    'Minh Khai - Bắc Từ Liêm',
    'Hà Nội: Chi cục BVMT (KK)',
    'Hà Nội: TT giao lưu văn hóa phố cổ - Hoàn Kiếm - Trạm cảm biến(KK)',
    'Công viên hồ điều hòa Nhân Chính, Khuất Duy Tiến',
    'Office IQAir Ha Noi',
    # 'IQAir Ha Noi',
    # 'FPT'
]

In [ ]:
df_FPT = df[df['station_name'] ==  'FPT']

df_FPT.head(1)

In [ ]:
df_BK = df[df['station_name'] ==  'Hà Nội: Đại Học Bách Khoa cổng Parabol đường Giải Phóng (KK)']
df_BK.head()

In [ ]:
df_BK_check = df_BK[df_BK['date'] >  datetime.date(2025,8,1)]

In [ ]:
df_BK_check.info()

## AQI/PM2.5 vs other index

In [ ]:
def compare_aqi_other(station, df, pollution= 'PM2.5 (µg/m³)', index='humidity (%)'):
    # Lọc dữ liệu của trạm đó
    station_data = df[df['station_name'] == station].sort_values('timestamp')
    
    # Tạo khung hình (figure) và trục thứ nhất (ax1)
    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    # Vẽ PM2.5 lên trục trái (Màu đỏ)
    color = 'tab:red'
    ax1.set_xlabel('Thời gian')
    ax1.set_ylabel('PM2.5', color=color)
    ax1.plot(station_data['timestamp'], station_data[pollution], color=color, alpha=0.7, label=pollution)
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Tạo trục thứ hai (ax2) chung trục hoành với ax1
    ax2 = ax1.twinx()  
    
    # Vẽ Humidity lên trục phải (Màu xanh)
    color = 'tab:blue'
    ax2.set_ylabel('Humidity (%)', color=color)  
    ax2.plot(station_data['timestamp'], station_data[index], color=color, alpha=0.5, linestyle='--', label=index)
    ax2.tick_params(axis='y', labelcolor=color)
    
    plt.title(f'Biểu đồ tương quan {pollution} và {index} tại trạm: {station}')
    fig.tight_layout()  # Để không bị cắt chữ
    plt.show()

In [ ]:

# Vẽ cho từng trạm (Bạn có thể giới hạn số lượng trạm bằng stations[:3] nếu quá nhiều)
for station in unique_stations:
    compare_aqi_other(station, df, pollution="aqi", index='humidity (%)')
    # compare_aqi_other(station, df, pollution="aqi", index='temperature (°)')


## Biểu đồ AQI/PM2.5 theo thời gian trong ngày

In [ ]:
def plot_specific_day(df, station_name, target_date_str, pollution='PM2.5 (µg/m³)'):
    """
    Hàm vẽ biểu đồ PM2.5 cho một ngày cụ thể.
    target_date_str: định dạng 'YYYY-MM-DD' (ví dụ '2025-04-17')
    """
    
    # Lọc dữ liệu trạm và ngày
    # Chuyển đổi cột date sang string để so sánh cho chính xác
    day_data = df[(df['station_name'] == station_name) & 
                  (df['date'].astype(str) == target_date_str)].sort_values('hour')
    
    if day_data.empty:
        print(f"Không có dữ liệu cho ngày {target_date_str} tại trạm {station_name}")
        return

    # --- VẼ BIỂU ĐỒ ---
    plt.figure(figsize=(12, 6))
    
    # Vẽ đường PM2.5
    plt.plot(day_data['hour'], day_data[pollution], marker='o', linestyle='-', color='teal', linewidth=2, label=target_date_str)
    
    plt.title(f'Diễn biến PM2.5 trong ngày {target_date_str} - Trạm: {station_name}')
    plt.xlabel('Giờ trong ngày (0-23h)')
    plt.ylabel('Nồng độ PM2.5 (µg/m³)')
    plt.xticks(range(0, 24))
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.show()



In [ ]:
for station in unique_stations:
    sample_date = "2025-06-14" 
    print(f"Đang vẽ cho ngày: {sample_date}")

    plot_specific_day(df, station, sample_date, pollution='aqi')

### Theo means & std

In [ ]:
def plot_individual_station_patterns(df, station_name, mode='full', date_range=None):
    """
    Plot hourly AQI patterns for a specific station.
    
    Parameters:
    -----------
    df : DataFrame
        The full dataset
    station_name : str
        Name of the station to plot
    mode : str, default='full'
        'full' - use entire dataset
        'date_range' - use specific date range
    date_range : tuple of str, optional
        (start_date, end_date) in format 'YYYY-MM-DD'
        Required if mode='date_range'
    """
    station_data = df[df['station_name'] == station_name].copy()
    
    # Filter by date range if specified
    if mode == 'date_range':
        if date_range is None:
            raise ValueError("date_range must be provided when mode='date_range'")
        
        start_date, end_date = date_range
        start_date = pd.to_datetime(start_date).date()
        end_date = pd.to_datetime(end_date).date()
        # Ensure date column exists
        if 'date' not in station_data.columns:
            raise ValueError("DataFrame must have a 'date' column")
        
        station_data_crop = station_data[
            (station_data['date'] >= start_date) & 
            (station_data['date'] <= end_date)
        ]
        
        if len(station_data) == 0:
            print(f"No data found for {station_name} in date range {start_date} to {end_date}")
            return

    hourly_stats = station_data_crop.groupby('hour').agg({
        'aqi': ['mean', 'std', 'count'],
        'PM2.5 (µg/m³)': ['mean', 'std', 'count']
    }).reset_index()
    
    hourly_stats.columns = ['hour', 'aqi_mean', 'aqi_std', 'aqi_count', 
                           'pm25_mean', 'pm25_std', 'pm25_count']
    
    # Fill NaN values
    hourly_stats['aqi_std'] = hourly_stats['aqi_std'].fillna(0)
    hourly_stats['pm25_std'] = hourly_stats['pm25_std'].fillna(0)
    
    fig, axes = plt.subplots(1, 1, figsize=(14, 10))
    
    # Create title suffix based on mode
    if mode == 'date_range':
        title_suffix = f' ({start_date} to {end_date})'
    else:
        title_suffix = ' (Full Dataset)'
    
    # Plot AQI
    axes.plot(hourly_stats['hour'], hourly_stats['aqi_mean'], 
                'o-', linewidth=3, markersize=8, color='blue', label='Mean AQI')
    
    if hourly_stats['aqi_count'].min() > 1:
        axes.fill_between(hourly_stats['hour'],
                           hourly_stats['aqi_mean'] - hourly_stats['aqi_std'],
                           hourly_stats['aqi_mean'] + hourly_stats['aqi_std'],
                           alpha=0.3, color='blue', label='±1 Std Dev')
    
    axes.set_title(f'Hourly AQI Patterns - {station_name}{title_suffix}', 
                     fontweight='bold', fontsize=14)
    axes.set_ylabel('AQI')
    axes.legend()
    axes.grid(True, alpha=0.3)
    axes.set_xlim(0, 23)
    axes.set_xticks(range(0, 24, 2))
    
    # Plot PM2.5 
    # axes[1].plot(hourly_stats['hour'], hourly_stats['pm25_mean'], 
    #             's-', linewidth=3, markersize=8, color='purple', label='Mean PM2.5')
    
    # if hourly_stats['pm25_count'].min() > 1:
    #     axes[1].fill_between(hourly_stats['hour'],
    #                        hourly_stats['pm25_mean'] - hourly_stats['pm25_std'],
    #                        hourly_stats['pm25_mean'] + hourly_stats['pm25_std'],
    #                        alpha=0.3, color='purple', label='±1 Std Dev')
    
    # axes[1].set_title(f'Hourly PM2.5 Patterns - {station_name}{title_suffix}', 
    #                  fontweight='bold', fontsize=14)
    # axes[1].set_xlabel('Hour of Day')
    # axes[1].set_ylabel('PM2.5 (µg/m³)')
    # axes[1].legend()
    # axes[1].grid(True, alpha=0.3)
    # axes[1].set_xlim(0, 23)
    # axes[1].set_xticks(range(0, 24, 2))
    
    plt.tight_layout()
    plt.show()

In [ ]:
for i, station in enumerate(unique_stations, 1):
    print(f"{station}")
    plot_individual_station_patterns(df, station, mode='date_range', date_range=('2025-6-01','2025-7-01'))

### AQI Hanoi

In [ ]:
print(len(hanoi_stations))

In [ ]:
# Filter and concatenate data
hanoi_df_list = []

for station in hanoi_stations:
    station_data = df[df['station_name'] == station].copy()
    if len(station_data) > 0:
        station_recent = station_data.sort_values('timestamp').tail(307)
        hanoi_df_list.append(station_recent)

hanoi_df = pd.concat(hanoi_df_list, ignore_index=True)
hanoi_df = hanoi_df.sort_values('timestamp')


def plot_hanoi_time_series():
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 12))

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', "#D66DAA", "#948d30", "#070505", "#7a06e7", "#2796d6"]  # Define colors for each station

    # AQI Plot
    for i, station in enumerate(hanoi_stations):
        station_data = hanoi_df[hanoi_df['station_name'] == station]

        ax1.plot(
            station_data['timestamp'],
            station_data['aqi'],
            label=station,
            color=colors[i],
            linewidth=2,
            alpha=0.9
        )

    ax1.set_title('AQI - All Hanoi Stations', fontsize=18, fontweight='bold')
    ax1.set_ylabel('AQI', fontsize=14)
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(axis='x', rotation=45)

    # PM2.5 Plot
    for i, station in enumerate(hanoi_stations):
        station_data = hanoi_df[hanoi_df['station_name'] == station]

        ax2.plot(
            station_data['timestamp'],
            station_data['PM2.5 (µg/m³)'],
            label=station,
            color=colors[i],
            linewidth=2,
            alpha=0.9
        )

    ax2.set_title('PM2.5 - All Hanoi Stations', fontsize=18, fontweight='bold')
    ax2.set_xlabel('Time', fontsize=14)
    ax2.set_ylabel('PM2.5 (µg/m³)', fontsize=14)
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

# Call the plotting function
plot_hanoi_time_series()